# MAJ-Debate: Stage 4 Judge Brain

This notebook consumes split-specific Stage 3 outputs, generates graph-constrained judgments, and writes split-specific Stage 4 outputs.


## 0. Imports & Configuration


In [2]:
import os, json, time, statistics
from pathlib import Path
from datetime import datetime
import requests
from dotenv import load_dotenv


cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'notebooks').exists()), cwd)
ENV_PATH = PROJECT_ROOT / '.env'
load_dotenv(ENV_PATH if ENV_PATH.exists() else None)

OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError('OPENROUTER_API_KEY is required for Stage 4.')
OPENROUTER_MODEL = os.environ.get('MAJ_STAGE4_MODEL', os.environ.get('MAJ_OPENROUTER_MODEL', 'google/gemma-3-12b-it'))
OPENROUTER_URL = os.environ.get('MAJ_OPENROUTER_URL', 'https://openrouter.ai/api/v1/chat/completions')
OPENROUTER_SITE_URL = os.environ.get('OPENROUTER_SITE_URL', 'http://localhost')
OPENROUTER_APP_NAME = os.environ.get('OPENROUTER_APP_NAME', 'MAJ-Debate-Stage4')

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://YOUR_VM_IP:5000')
MODEL = OPENROUTER_MODEL
TEMPERATURE = 0.2
MAX_TOKENS = 500
RATE_LIMIT_SLEEP = 1.2
TOPIC_LIMIT = int(os.environ.get('MAJ_STAGE4_TOPIC_LIMIT', '0'))
EVAL_SPLIT = os.environ.get('MAJ_EVAL_SPLIT', 'ddo_sample')
RESUME_EXISTING = os.environ.get('MAJ_STAGE4_RESUME', '1').strip() not in {'0', 'false', 'False'}
CONTINUE_ON_ERROR = os.environ.get('MAJ_STAGE4_CONTINUE_ON_ERROR', '1').strip() not in {'0', 'false', 'False'}
MAX_RETRIES = 3

IN_FILE = PROJECT_ROOT / 'outputs' / 'stage3' / EVAL_SPLIT / 'stage3_graphs.json'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'stage4' / EVAL_SPLIT
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / 'stage4_judgments.json'
FAILURES_FILE = OUT_DIR / 'stage4_failures.json'
print(f'.env    : {ENV_PATH if ENV_PATH.exists() else "not found"}')
print('Provider : openrouter')
print(f'OpenRouter URL : {OPENROUTER_URL}')
print(f'Model   : {MODEL}')
print(f'Split   : {EVAL_SPLIT}')
print(f'Input   : {IN_FILE.resolve()}')
print(f'Output  : {OUT_FILE.resolve()}')
print(f'Failures: {FAILURES_FILE.resolve()}')
print(f'Resume  : {RESUME_EXISTING}')
print(f'Continue: {CONTINUE_ON_ERROR}')


.env    : /home/jupyter-st125974-ml/Project/.env
Provider : openrouter
OpenRouter URL : https://openrouter.ai/api/v1/chat/completions
Model   : google/gemma-3-12b-it
Split   : ddo_sample
Input   : /home/jupyter-st125974-ml/Project/outputs/stage3/ddo_sample/stage3_graphs.json
Output  : /home/jupyter-st125974-ml/Project/outputs/stage4/ddo_sample/stage4_judgments.json
Failures: /home/jupyter-st125974-ml/Project/outputs/stage4/ddo_sample/stage4_failures.json
Resume  : True
Continue: True


## 1. Prompt Template and Core Functions


In [3]:
def build_judge_prompt(topic_rec):
    arg_index = {a['arg_id']: a for a in topic_rec['arguments']}
    ext = topic_rec['grounded_extension'] if topic_rec['winner'] != 'UNDECIDED' else (topic_rec['preferred_extensions'][0] if topic_rec['preferred_extensions'] else [])
    selected = [arg_index[arg_id] for arg_id in ext if arg_id in arg_index]
    decisive = topic_rec.get('decisive_attacks', [])[:5]
    selected_block = '\n'.join(f"- {a['arg_id']} ({a['stance']}): {a['text']}" for a in selected) or '- None'
    attack_block = '\n'.join(f"- {e['source_arg_id']} attacks {e['target_arg_id']}: {e['justification']}" for e in decisive) or '- None'
    return f"""You are the MAJ-Debate Judge Brain. The graph winner is a hard constraint.\n\nTopic: {topic_rec['topic_text']}\nWinner side (must not change): {topic_rec['winner']}\n\nSelected extension arguments:\n{selected_block}\n\nDecisive attacks:\n{attack_block}\n\nTask:\n1. Explain why the graph-selected side prevails.\n2. Cite specific selected arguments.\n3. Identify decisive killing attacks.\n4. Assign a persuasiveness rating from 1 to 5.\n5. Assign a confidence value from 0 to 1.\n\nReturn JSON only:\n{{"winner": "PRO", "explanation": "...", "decisive_attacks": ["A001 attacks A010"], "persuasiveness": 4, "confidence": 0.79}}"""

NO_THINKING_INSTRUCTION = (
    'Do not think step by step. Do not include chain-of-thought, analysis, or commentary. '
    'Return only the requested JSON and nothing else.'
)

def call_openrouter(prompt):
    headers = {
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        'Content-Type': 'application/json',
        'HTTP-Referer': OPENROUTER_SITE_URL,
        'X-Title': OPENROUTER_APP_NAME,
    }
    payload = {
        'model': MODEL,
        'messages': [
            {'role': 'system', 'content': NO_THINKING_INSTRUCTION},
            {'role': 'user', 'content': prompt},
        ],
        'temperature': TEMPERATURE,
        'max_tokens': MAX_TOKENS,
        'reasoning': {'exclude': True, 'enabled': False},
    }
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=120)
        except requests.RequestException as ex:
            last_error = ex
            time.sleep(min(2 ** attempt, 30))
            continue
        if resp.status_code == 429:
            retry_after = resp.headers.get('Retry-After')
            try:
                wait_s = float(retry_after) if retry_after else 5 * attempt
            except ValueError:
                wait_s = 5 * attempt
            last_error = RuntimeError(f'OpenRouter 429 rate limit (retry after {wait_s}s): {resp.text[:200]}')
            time.sleep(wait_s)
            continue
        if resp.status_code in (500, 502, 503, 504):
            last_error = RuntimeError(f'OpenRouter {resp.status_code}: {resp.text[:200]}')
            time.sleep(min(2 ** attempt, 30))
            continue
        if resp.status_code != 200:
            raise RuntimeError(
                f'OpenRouter API error {resp.status_code}: {resp.text[:500]}. '
                f'Check model slug ({MODEL!r}), API key, and account credits.'
            )
        data = resp.json()
        choices = data.get('choices') or []
        if not choices:
            last_error = RuntimeError(f'OpenRouter returned no choices: {data}')
            time.sleep(RATE_LIMIT_SLEEP * attempt)
            continue
        message = choices[0].get('message', {})
        content = message.get('content', '')
        if isinstance(content, list):
            content = ''.join(part.get('text', '') for part in content if isinstance(part, dict))
        content = (content or '').strip()
        if content:
            return content
        last_error = RuntimeError(f'OpenRouter returned empty content: {data}')
        time.sleep(RATE_LIMIT_SLEEP * attempt)
    raise RuntimeError(f'OpenRouter call failed after {MAX_RETRIES} attempts: {last_error}')

def parse_json_object(text):
    text = text.strip()
    if '```' in text:
        parts = text.split('```')
        if len(parts) >= 3:
            text = parts[1].replace('json', '').strip()
    s = text.find('{')
    e = text.rfind('}') + 1
    return json.loads(text[s:e])

def load_stage3(path=IN_FILE):
    if not path.exists():
        raise FileNotFoundError(f'Stage 3 input not found: {path}')
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def selected_extension(topic_rec):
    if topic_rec['winner'] != 'UNDECIDED':
        return topic_rec.get('grounded_extension', [])
    pref = topic_rec.get('preferred_extensions', [])
    return pref[0] if pref else []

def judge_topic(topic_rec):
    raw = call_openrouter(build_judge_prompt(topic_rec))
    time.sleep(RATE_LIMIT_SLEEP)
    data = parse_json_object(raw)
    stated_winner = data.get('winner', 'UNDECIDED')
    return {'topic_id': topic_rec['topic_id'], 'topic_text': topic_rec['topic_text'], 'domain': topic_rec.get('domain'), 'benchmark_label': topic_rec.get('benchmark_label'), 'source_dataset': topic_rec.get('source_dataset'), 'source_ref': topic_rec.get('source_ref'), 'evaluation_split': topic_rec.get('evaluation_split', EVAL_SPLIT), 'graph_winner': topic_rec['winner'], 'judge_winner': stated_winner, 'selected_extension': selected_extension(topic_rec), 'explanation': data.get('explanation', ''), 'decisive_attacks': data.get('decisive_attacks', []), 'persuasiveness': int(data.get('persuasiveness', 3)), 'confidence': round(float(data.get('confidence', 0.5)), 3), 'hallucination_conflict': stated_winner != topic_rec['winner'], 'source_graph_summary': topic_rec.get('summary', {})}

try:
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment('MAJ-Debate')
    MLFLOW_OK = True
except Exception as ex:
    MLFLOW_OK = False
    print(f'MLflow unavailable ({ex}) - results saved locally only')

def mlflow_log(run_name, params, metrics, artifact_paths):
    if not MLFLOW_OK:
        return
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        for path in artifact_paths:
            mlflow.log_artifact(str(path), artifact_path=f'stage4/{EVAL_SPLIT}')


MLflow unavailable (No module named 'mlflow') - results saved locally only


## 2. Main Run - Stage 4 on All Topics


In [4]:
stage3_doc = load_stage3()
topics = stage3_doc['topics'][:TOPIC_LIMIT] if TOPIC_LIMIT > 0 else stage3_doc['topics']
run_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
run_name = f'stage4-{EVAL_SPLIT}-{run_ts}'

def compute_stage4_summary(topic_payloads):
    return {'total_topics': len(topic_payloads), 'hallucination_conflicts': sum(1 for t in topic_payloads if t['hallucination_conflict']), 'avg_persuasiveness': round(statistics.mean(t['persuasiveness'] for t in topic_payloads), 3) if topic_payloads else 0.0, 'avg_confidence': round(statistics.mean(t['confidence'] for t in topic_payloads), 3) if topic_payloads else 0.0}

def save_stage4_state(topic_results_by_id, failures):
    ordered_topics = [topic_results_by_id[t['topic_id']] for t in topics if t['topic_id'] in topic_results_by_id]
    output_doc = {'stage': 4, 'run_name': run_name, 'timestamp': run_ts, 'config': {'provider': 'openrouter', 'model': MODEL, 'openrouter_url': OPENROUTER_URL, 'evaluation_split': EVAL_SPLIT, 'source_stage3_run': stage3_doc.get('run_name'), 'resume_existing': RESUME_EXISTING}, 'topics': ordered_topics, 'summary': compute_stage4_summary(ordered_topics)}
    with open(OUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(output_doc, f, indent=2)
    with open(FAILURES_FILE, 'w', encoding='utf-8') as f:
        json.dump({'run_name': run_name, 'timestamp': run_ts, 'failed_topics': failures}, f, indent=2)
    return output_doc

existing_topics_by_id = {}
if RESUME_EXISTING and OUT_FILE.exists():
    try:
        with open(OUT_FILE, 'r', encoding='utf-8') as f:
            existing_doc = json.load(f)
        for topic_payload in existing_doc.get('topics', []):
            topic_id = topic_payload.get('topic_id')
            if topic_id and topic_payload.get('explanation'):
                existing_topics_by_id[topic_id] = topic_payload
        print(f'Resuming from existing output: {len(existing_topics_by_id)} completed topics found')
    except Exception as ex:
        print(f'Could not resume from existing output ({ex}); starting fresh')
        existing_topics_by_id = {}

existing_failures_by_id = {}
if FAILURES_FILE.exists():
    try:
        with open(FAILURES_FILE, 'r', encoding='utf-8') as f:
            existing_failures_doc = json.load(f)
        for failure_payload in existing_failures_doc.get('failed_topics', []):
            failed_topic_id = failure_payload.get('topic_id')
            if failed_topic_id and failed_topic_id not in existing_topics_by_id:
                existing_failures_by_id[failed_topic_id] = failure_payload
    except Exception as ex:
        print(f'Could not load existing failures ({ex}); continuing with a fresh failure log')
        existing_failures_by_id = {}

topic_results_by_id = dict(existing_topics_by_id)
failures_by_id = dict(existing_failures_by_id)
for idx, topic in enumerate(topics, start=1):
    topic_id = topic['topic_id']
    if topic_id in topic_results_by_id:
        print(f"[{idx}/{len(topics)}] {topic_id}: already completed, skipping")
        continue
    print(f"[{idx}/{len(topics)}] {topic_id}: {topic['topic_text']}")
    try:
        topic_results_by_id[topic_id] = judge_topic(topic)
        failures_by_id.pop(topic_id, None)
        output_doc = save_stage4_state(topic_results_by_id, list(failures_by_id.values()))
        print(f"  saved checkpoint after {topic_id}")
    except Exception as ex:
        failures_by_id[topic_id] = {'topic_id': topic_id, 'topic_text': topic['topic_text'], 'error': str(ex), 'run_name': run_name}
        output_doc = save_stage4_state(topic_results_by_id, list(failures_by_id.values()))
        if not CONTINUE_ON_ERROR:
            raise

output_doc = save_stage4_state(topic_results_by_id, list(failures_by_id.values()))
mlflow_log(run_name=run_name, params=output_doc['config'], metrics={k: float(v) for k, v in output_doc['summary'].items() if isinstance(v, (int, float))}, artifact_paths=[OUT_FILE, FAILURES_FILE])
print(f'Saved: {OUT_FILE.resolve()}')
print(output_doc['summary'])
print(f'Failed topics: {len(failures_by_id)}')


[1/500] DDO_00423: A Ban is an Act of War.
  saved checkpoint after DDO_00423
[2/500] DDO_00430: A beggar is smarter than a person with PHD
  saved checkpoint after DDO_00430
[3/500] DDO_00431: A Being Cannot be Both Omnipotent and Omniscient
  saved checkpoint after DDO_00431
[4/500] DDO_00437: A bigger military enforces more individual rights.
  saved checkpoint after DDO_00437
[5/500] DDO_00456: A bird in the hand is worth two in the bush.
  saved checkpoint after DDO_00456
[6/500] DDO_00460: a blind man seeing aliens is a blind man not seeing aliens
  saved checkpoint after DDO_00460
[7/500] DDO_00484: A Chernobyl-type nuclear power accident cannot happen in the United States
  saved checkpoint after DDO_00484
[8/500] DDO_00519: A Comprehensive Immigration Reform is key to our Economy and one should be issued
  saved checkpoint after DDO_00519
[9/500] DDO_00522: A Conscious Mind, Based On Everything We Know, Is Probably The Basis And Grounding Of The Universe
  saved checkpoint aft

## 3. Inspect Output


In [5]:
sample = output_doc['topics'][0]
print('Topic           :', sample['topic_text'])
print('Graph winner    :', sample['graph_winner'])
print('Judge winner    :', sample['judge_winner'])
print('Benchmark label :', sample['benchmark_label'])
print('Hallucination   :', sample['hallucination_conflict'])


Topic           : A Ban is an Act of War.
Graph winner    : PRO
Judge winner    : PRO
Benchmark label : CON
Hallucination   : False
